# R13 — multimodal embeddings на GPU

Блокнот выполняет сначала smoke-тест на 10 товарах, затем полный возобновляемый расчёт. Кеш и результаты сохраняются на Google Drive.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/ecup')
ARCHIVE = DRIVE_ROOT / 'input/images.zip'
DATA_CSV = DRIVE_ROOT / 'input/data.csv'
MANIFEST = DRIVE_ROOT / 'input/image_manifest.parquet'
CACHE_DIR = DRIVE_ROOT / 'cache/embeddings'
OUTPUT_DIR = DRIVE_ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for path in (ARCHIVE, DATA_CSV, MANIFEST):
    assert path.exists(), f'Не найден входной файл: {path}'
print('Входные файлы найдены')

In [ ]:
import torch

assert torch.cuda.is_available(), 'В Colab выберите Runtime → Change runtime type → GPU'
device = torch.cuda.get_device_properties(0)
gpu_gb = device.total_memory / 1024**3
BATCH_SIZE = 4 if gpu_gb >= 35 else (2 if gpu_gb >= 20 else 1)
print(device.name, f'{gpu_gb:.1f} GB', f'batch_size={BATCH_SIZE}')
!nvidia-smi

In [ ]:
import os
import subprocess

REPO_DIR = Path('/content/quality-control')
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'feature/R13-multimodal-embeddings',
        'https://github.com/zimmer10/quality-control.git', str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
subprocess.run(['pip', 'install', '-q', '-e', '.[embeddings]'], check=True)
print('Зависимости установлены')

In [ ]:
IMAGES_PARENT = Path('/content/ecup_data')
IMAGES_ROOT = IMAGES_PARENT / 'images'
if not IMAGES_ROOT.exists():
    IMAGES_PARENT.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', str(ARCHIVE), '-d', str(IMAGES_PARENT)], check=True)
assert IMAGES_ROOT.exists(), f'После распаковки не найдена папка {IMAGES_ROOT}'
print('Изображения готовы:', IMAGES_ROOT)

In [ ]:
from huggingface_hub import snapshot_download

SHARED_MODELS = Path('/content/shared_models')
MODEL_DIR = SHARED_MODELS / 'Qwen/Qwen3-VL-Embedding-2B'
if not MODEL_DIR.exists():
    snapshot_download(
        repo_id='Qwen/Qwen3-VL-Embedding-2B',
        local_dir=MODEL_DIR,
    )
os.environ['SHARED_MODELS_PATH'] = str(SHARED_MODELS)
print('Модель готова:', MODEL_DIR)

In [ ]:
import time

SMOKE_OUTPUT = OUTPUT_DIR / 'embeddings-smoke.parquet'
SMOKE_REPORT = OUTPUT_DIR / 'R13-embeddings-smoke.md'
smoke_command = [
    'python', '-m', 'ecup.features.embeddings',
    '--data', str(DATA_CSV),
    '--manifest', str(MANIFEST),
    '--images-root', str(IMAGES_ROOT),
    '--cache-dir', str(CACHE_DIR),
    '--output', str(SMOKE_OUTPUT),
    '--report', str(SMOKE_REPORT),
    '--limit', '10',
    '--batch-size', str(BATCH_SIZE),
    '--progress-every', '1',
]
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src')
started = time.perf_counter()
subprocess.run(smoke_command, env=env, check=True)
SMOKE_SECONDS = time.perf_counter() - started
print(f'Smoke готов за {SMOKE_SECONDS:.1f} s:', SMOKE_OUTPUT)

In [ ]:
import polars as pl

smoke = pl.read_parquet(SMOKE_OUTPUT)
display(smoke.select(
    'id', 'embedding_status', 'source_image_count',
    'embedded_image_count', 'text_image_similarity', 'image_disagreement'
))
assert smoke.height == 10
assert smoke.filter(pl.col('embedding_status') == 'embedding_error').is_empty()
total_products = pl.scan_csv(DATA_CSV).select(pl.len()).collect().item()
estimated_hours = SMOKE_SECONDS / smoke.height * total_products / 3600
print(f'Грубая последовательная оценка полного запуска: {estimated_hours:.1f} h')

In [ ]:
FULL_OUTPUT = OUTPUT_DIR / 'embeddings.parquet'
FULL_REPORT = OUTPUT_DIR / 'R13-embeddings.md'
full_command = [
    'python', '-m', 'ecup.features.embeddings',
    '--data', str(DATA_CSV),
    '--manifest', str(MANIFEST),
    '--images-root', str(IMAGES_ROOT),
    '--cache-dir', str(CACHE_DIR),
    '--output', str(FULL_OUTPUT),
    '--report', str(FULL_REPORT),
    '--batch-size', str(BATCH_SIZE),
    '--progress-every', '100',
]
subprocess.run(full_command, env=env, check=True)
print('Полный результат:', FULL_OUTPUT)
print('Отчёт:', FULL_REPORT)